In [92]:
import json

# 假设你的文件名是data.json
file_path = '/p/lustre2/shan4/Fuzzlang/log_llvm_remove3.json'
# 初始化一个空列表来存储所有的JSON对象
json_objects = []

# 打开文件并逐行读取
with open(file_path, 'r', encoding='utf-8') as file:
    i = 1
    for line in file:
        # 解析每一行的JSON对象
        try:
            json_object = json.loads(line.strip())
        except:
            continue
        # 将解析的对象添加到列表中
        json_objects.append(json_object)
        i += 1
success = [i for i in json_objects if i['status']=='SUCCESS' ]
print(len(success))
failed = [i for i in json_objects if i['status']=='ERROR' ]
print(len(failed))

2895
6


In [9]:
def count_lines_in_file(file_path):
    try:
        with open(file_path, 'r') as file:
            lines = file.readlines()
            line_count = len(lines)
        return line_count
    except FileNotFoundError:
        return "File not found. Please check the file path."

# Replace 'your_file.txt' with the path to your text file
file_path = '../log_llvm_removept3.json'
line_count = count_lines_in_file(file_path)
print(f"The file '{file_path}' has {line_count} lines.")

The file '../log_llvm_removept3.json' has 17240 lines.


In [1]:
import clang.cindex

def list_cursor_kinds():
    kinds = []
    for kind in clang.cindex.CursorKind.get_all_kinds():
        kinds.append((kind, kind.name, kind.value))
    return kinds

if __name__ == "__main__":
    cursor_kinds = list_cursor_kinds()
    for kind, name, value in cursor_kinds:
        print(f"Kind: {kind}, Name: {name}, Value: {value}")

Kind: CursorKind.UNEXPOSED_DECL, Name: UNEXPOSED_DECL, Value: 1
Kind: CursorKind.STRUCT_DECL, Name: STRUCT_DECL, Value: 2
Kind: CursorKind.UNION_DECL, Name: UNION_DECL, Value: 3
Kind: CursorKind.CLASS_DECL, Name: CLASS_DECL, Value: 4
Kind: CursorKind.ENUM_DECL, Name: ENUM_DECL, Value: 5
Kind: CursorKind.FIELD_DECL, Name: FIELD_DECL, Value: 6
Kind: CursorKind.ENUM_CONSTANT_DECL, Name: ENUM_CONSTANT_DECL, Value: 7
Kind: CursorKind.FUNCTION_DECL, Name: FUNCTION_DECL, Value: 8
Kind: CursorKind.VAR_DECL, Name: VAR_DECL, Value: 9
Kind: CursorKind.PARM_DECL, Name: PARM_DECL, Value: 10
Kind: CursorKind.OBJC_INTERFACE_DECL, Name: OBJC_INTERFACE_DECL, Value: 11
Kind: CursorKind.OBJC_CATEGORY_DECL, Name: OBJC_CATEGORY_DECL, Value: 12
Kind: CursorKind.OBJC_PROTOCOL_DECL, Name: OBJC_PROTOCOL_DECL, Value: 13
Kind: CursorKind.OBJC_PROPERTY_DECL, Name: OBJC_PROPERTY_DECL, Value: 14
Kind: CursorKind.OBJC_IVAR_DECL, Name: OBJC_IVAR_DECL, Value: 15
Kind: CursorKind.OBJC_INSTANCE_METHOD_DECL, Name: OBJC_I

In [86]:
print(failed[10]['fuzzed_args'])

['-c', '-fno-exceptions', '-D_GNU_SOURCE', '-o lib/Support/CMakeFiles/LLVMSupport.dir/Base64.cpp.o', '-Wctad-maybe-unsupported', '-Wnon-virtual-dtor', '-Werror=global-constructors', '-Wdelete-non-virtual-dtor', '-Wstring-conversion', '-I/p/lustre2/shan4/llvm-project/llvm/lib/Support']


In [85]:
print(failed[10]['message'])

/opt/rh/gcc-toolset-12/root/usr/lib/gcc/x86_64-redhat-linux/12/../../../../bin/ld: /lib/../lib64/Scrt1.o: in function `_start':
(.text+0x24): undefined reference to `main'
/opt/rh/gcc-toolset-12/root/usr/lib/gcc/x86_64-redhat-linux/12/../../../../bin/ld: /var/tmp/shan4/Base64-54ab22.o: in function `llvm::createStringError(std::error_code, char const*)':
Base64.cpp:(.text._ZN4llvm17createStringErrorESt10error_codePKc[_ZN4llvm17createStringErrorESt10error_codePKc]+0xa2): undefined reference to `llvm::createStringError(std::__cxx11::basic_string<char, std::char_traits<char>, std::allocator<char> >&&, std::error_code)'
/opt/rh/gcc-toolset-12/root/usr/lib/gcc/x86_64-redhat-linux/12/../../../../bin/ld: /var/tmp/shan4/Base64-54ab22.o: in function `llvm::Error llvm::createStringError<char, unsigned long>(std::error_code, char const*, char const&, unsigned long const&)':
Base64.cpp:(.text._ZN4llvm17createStringErrorIJcmEEENS_5ErrorESt10error_codePKcDpRKT_[_ZN4llvm17createStringErrorIJcmEEENS_5E

In [179]:
from clang.cindex import Index, Cursor, CursorKind, SourceLocation, TokenKind
from clang.cindex import Index, Cursor, CursorKind, SourceLocation


def get_logical_line_at_offset(filename, offset):
    index = Index.create()
    tu = index.parse(filename)
    
    file = tu.get_file(filename)
    location = SourceLocation.from_offset(tu, file, offset)

    cursor = Cursor.from_location(tu, location)

    while cursor and cursor.kind == CursorKind.UNEXPOSED_EXPR:
        cursor = cursor.semantic_parent

    if cursor and cursor.kind in [CursorKind.FUNCTION_DECL, CursorKind.CXX_METHOD, CursorKind.CONSTRUCTOR, CursorKind.DESTRUCTOR, CursorKind.CLASS_DECL]:
        tokens = list(cursor.get_tokens())
        extent = cursor.extent
        
        start_line = extent.start.line
        end_line = extent.end.line
        for i, token in enumerate(tokens):
            if token.kind == TokenKind.PUNCTUATION and token.spelling == '{':
                end_line = tokens[i-1].extent.end.line
                print(end_line)
                break
        with open(filename, 'r') as f:
            content = f.read()
            print(content[offset])
        with open(filename, 'r') as f:
            lines = f.readlines()
            print("here")
            print(cursor.kind)
            print(''.join(lines[start_line-1:end_line]).strip())
            return ''.join(lines[start_line-1:end_line]).strip()

    if cursor:
        extent = cursor.extent
        start_line = extent.start.line
        end_line = extent.end.line
        print(f"start line {start_line}")
        print(f"end line {end_line}")
        with open(filename, 'r') as f:
            content = f.read()
            print(content[offset])
        with open(filename, 'r') as f:
            lines = f.readlines()
            print("here1")
            print(cursor.kind)
            print(''.join(lines[start_line-1:end_line]).strip())
            return ''.join(lines[start_line:end_line-1]).strip()

    return None
get_logical_line_at_offset("../tmp.cpp", 136)
ttt = 1

start line 5
end line 8
r
here1
CursorKind.COMPOUND_STMT
~tmp(){
            std::cout << "tmp destructor" << std::endl;
            int a = 5;
        }


In [25]:
get_logical_line_at_offset("../tmp.cpp", 105)

'std::cout << "tmp constructor" << std::endl;'

In [117]:
def get_line_at_offset(content, offset):
    start = content.rfind('\n', 0, offset) + 1
    end = content.find('\n', offset)
    if end == -1:
        end = len(content)

    # Check for line continuation
    while content[end-1] == '\\':
        next_end = content.find('\n', end + 1)
        if next_end == -1:
            end = len(content)
            break
        end = next_end

    return content[start:end]

In [58]:
with open("../tmp.cpp", 'r') as file:
        content = file.read()
get_line_at_offset(content, 230)


'void func(){'

In [80]:
get_declaration_at_offset("../tmp.cpp", 220)

In [1]:
import clang.cindex


def print_ast(node, indent=0):
    print('  ' * indent + str(node.kind), node.spelling)
    for child in node.get_children():
        print_ast(child, indent + 1)


if not clang.cindex.Config.library_file:
    clang.cindex.Config.set_library_file(
        '/p/lustre2/shan4/llvm-trunk/lib/libclang.so')
    
    
index = clang.cindex.Index.create()


cpp_code = """
struct tttt{
    int a;
    int b;
};

int main() {
    struct tttt ccc;
    tttt ccc;
    return 0;
}
"""
tu = index.parse('temp.c', unsaved_files=[('temp.c', cpp_code)])

print_ast(tu.cursor)

CursorKind.TRANSLATION_UNIT temp.c
  CursorKind.STRUCT_DECL tttt
    CursorKind.FIELD_DECL a
    CursorKind.FIELD_DECL b
  CursorKind.FUNCTION_DECL main
    CursorKind.COMPOUND_STMT 
      CursorKind.DECL_STMT 
        CursorKind.VAR_DECL ccc
          CursorKind.TYPE_REF struct tttt
      CursorKind.DECL_STMT 
        CursorKind.VAR_DECL ccc
          CursorKind.TYPE_REF struct tttt
      CursorKind.RETURN_STMT 
        CursorKind.INTEGER_LITERAL 


In [3]:
import clang.cindex

def get_cursor_kind_name(kind):
    return str(kind).split('.')[-1]

def find_parentheses(node, filename, source):
    if node.location.file and node.location.file.name != filename:
        return

    if node.kind == clang.cindex.CursorKind.CALL_EXPR or node.kind == clang.cindex.CursorKind.FUNCTION_DECL:
        start = node.extent.start.offset
        end = node.extent.end.offset
        content = source[start:end]
        
        # 查找左括号的位置
        left_paren_pos = content.find('(')
        if left_paren_pos != -1:
            abs_left_paren_pos = start + left_paren_pos
            print(f"Left parenthesis at position {abs_left_paren_pos}, type: {get_cursor_kind_name(node.kind)}")

        # 查找右括号的位置
        right_paren_pos = content.rfind(')')
        if right_paren_pos != -1:
            abs_right_paren_pos = start + right_paren_pos
            print(f"Right parenthesis at position {abs_right_paren_pos}, type: {get_cursor_kind_name(node.kind)}")

    for child in node.get_children():
        find_parentheses(child, filename, source)

def main():
    index = clang.cindex.Index.create()
    
    # 替换为您的C/C++文件路径
    file_path = 'tmp.cpp'
    
    with open(file_path, 'r') as file:
        source = file.read()

    tu = index.parse(file_path)
    
    if not tu:
        print("Failed to parse the source file.")
        return

    find_parentheses(tu.cursor, file_path, source)

if __name__ == "__main__":
    main()

Left parenthesis at position 25, type: FUNCTION_DECL
Right parenthesis at position 26, type: FUNCTION_DECL
Left parenthesis at position 90, type: FUNCTION_DECL
Right parenthesis at position 240, type: FUNCTION_DECL
Left parenthesis at position 102, type: CALL_EXPR
Right parenthesis at position 103, type: CALL_EXPR
Left parenthesis at position 116, type: CALL_EXPR
Right parenthesis at position 124, type: CALL_EXPR
Left parenthesis at position 137, type: CALL_EXPR
Right parenthesis at position 145, type: CALL_EXPR
Left parenthesis at position 158, type: CALL_EXPR
Right parenthesis at position 166, type: CALL_EXPR
Left parenthesis at position 179, type: CALL_EXPR
Right parenthesis at position 187, type: CALL_EXPR
Left parenthesis at position 200, type: CALL_EXPR
Right parenthesis at position 208, type: CALL_EXPR
Left parenthesis at position 218, type: CALL_EXPR
Right parenthesis at position 219, type: CALL_EXPR
Left parenthesis at position 232, type: CALL_EXPR
Right parenthesis at positio

In [6]:
from clang.cindex import CursorKind

def print_cursor_kinds():
    for attr in dir(CursorKind):
        if not attr.startswith("__"):  # 过滤掉内置属性
            value = getattr(CursorKind, attr)
            if isinstance(value, CursorKind):
                print(f"{attr}: {value}")

In [12]:
import clang.cindex
from clang.cindex import CursorKind, TokenKind

def get_cursor_kind_name(kind):
    return str(kind).split('.')[-1]

def find_parenthesis_attribute(tu, file_path, line, column):
    def get_cursor_at_location(cursor, line, column):
        if (cursor.location.file and 
            cursor.location.file.name == file_path and
            cursor.extent.start.line <= line <= cursor.extent.end.line and
            cursor.extent.start.column <= column <= cursor.extent.end.column):
            
            for child in cursor.get_children():
                result = get_cursor_at_location(child, line, column)
                if result:
                    return result
            return cursor
        return None

    cursor = get_cursor_at_location(tu.cursor, line, column)
    if not cursor:
        return "No cursor found at the specified location."

    # 检查是否确实是括号
    for token in cursor.get_tokens():
        if (token.location.line == line and 
            token.location.column == column and 
            token.kind == TokenKind.PUNCTUATION and 
            token.spelling in '()'):
            break
    else:
        return "No parenthesis found at the specified location."

    # 向上遍历AST找到最近的有意义的父节点
    while cursor and cursor.kind in [CursorKind.UNEXPOSED_EXPR, CursorKind.DECL_REF_EXPR]:
        cursor = cursor.semantic_parent

    if not cursor:
        return "Could not determine the context of the parenthesis."

    return f"Parenthesis at {file_path}:{line}:{column} is part of a {get_cursor_kind_name(cursor.kind)}"

def main():
    index = clang.cindex.Index.create()
    
    # 替换为您的C/C++文件路径
    file_path = 'tmp.cpp'
    
    tu = index.parse(file_path)
    
    if not tu:
        print("Failed to parse the source file.")
        return

    # 替换为您知道的括号位置
    line = 10
    column = 15

    result = find_parenthesis_attribute(tu, file_path, line, column)
    print(result)

if __name__ == "__main__":
    main()

No cursor found at the specified location.
